# Drone Detection — v5 (DJI-Fix Fine-Tune)

v4 (`yolo_finetune_recall_v4`) pushed recall aggressively but **regressed DJI detection** — DJI is now sometimes labeled as **Bird**.

Two suspected causes:

1. **Class-ID mismatch** between the dataset's `data.yaml` and the runtime model. Backend logs confirm the runtime mapping is:
   ```
   0: Bird   1: shahed_136   2: orlan   3: Airplane   4: Helicopter   5: dji
   ```
   An older notebook (`drone_trackFinal.ipynb`) used a *different* order. If the dataset's `data.yaml` still has the old order, fine-tuning from `best.pt` will silently swap labels.
2. **Recall-aggressive augmentation blurring Bird ↔ DJI.** Mixup interpolates labels; with `cls=0.4` (lowered) and `mixup=0.20`, two small-aerial-silhouette classes collapse into each other.

This notebook:

- §4 **audits the dataset** — class counts, names integrity, suspicious tiny-bbox ratios.
- §5 **auto-fixes the class-ID mapping** if the dataset's order doesn't match the runtime.
- §6 **oversamples DJI** if it's underrepresented relative to Bird.
- §8 **trains with class-separation hyperparameters** — `mixup=0`, `copy_paste=0.05`, `cls=0.6`. Keeps the recall wins from v4 (`imgsz=1280`, `box=8.0`) without the label-blurring tricks.
- §9 reports **per-class** precision/recall so DJI is graded explicitly, not buried in the mean.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q "ultralytics>=8.3.0" "albumentations<2.0" pyyaml seaborn

## 2. Configuration

In [ ]:
from pathlib import Path

# --- Source data ---
# Source dataset on Drive. The user renamed it from final_data after
# adding more samples. The §3 unpack logic probes both <name> and
# <name>.zip and accepts either a folder or a zip — works either way.
DRIVE_DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/data_added_final"
WORK_DATA_DIR   = Path("/content/dataset")

# --- Model ---
BASE_MODEL = "yolo26s.pt"
PREV_BEST_PT = "/content/drive/MyDrive/Colab Notebooks/final_model/best.pt"

# --- Authoritative class order (matches the backend runtime) ---
# Backend log: 'Model classes: {0: Bird, 1: shahed_136, 2: orlan,
#                              3: Airplane, 4: Helicopter, 5: dji}'
# This is the source of truth — if the dataset disagrees, we remap.
RUNTIME_CLASS_NAMES = ["Bird", "shahed_136", "orlan", "Airplane", "Helicopter", "dji"]
DJI_CLASS_ID = RUNTIME_CLASS_NAMES.index("dji")  # 5

# Oversample DJI if its label-file count is below this fraction of the
# largest class. 0.5 keeps DJI within a factor of 2 of Bird etc.
DJI_OVERSAMPLE_TARGET_RATIO = 0.5

# --- Training (class-separation leaning) ---
EPOCHS       = 100
IMG_SIZE     = 1280     # keep — biggest small-target recall win
BATCH        = 8
SAVE_PERIOD  = 5
PATIENCE     = 30
RUN_NAME     = "drone_finetune_v5_dji_fix"

RESUME = False  # fresh start — we want the new label mapping in effect from epoch 0

# --- Output ---
DRIVE_ROOT   = Path("/content/drive/MyDrive/Colab Notebooks/final_model")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("config loaded — runs land in", DRIVE_ROOT / "runs")
print("RUNTIME_CLASS_NAMES =", RUNTIME_CLASS_NAMES)

## 3. Unpack the dataset

In [ ]:
import shutil, zipfile

src = Path(DRIVE_DATA_PATH)
zip_candidate = Path(str(DRIVE_DATA_PATH) + (".zip" if not str(DRIVE_DATA_PATH).endswith(".zip") else ""))

if WORK_DATA_DIR.exists():
    shutil.rmtree(WORK_DATA_DIR)
WORK_DATA_DIR.mkdir(parents=True)

if src.is_file() and src.suffix.lower() == ".zip":
    with zipfile.ZipFile(src) as z: z.extractall(WORK_DATA_DIR)
elif src.is_dir():
    shutil.copytree(src, WORK_DATA_DIR, dirs_exist_ok=True)
elif zip_candidate.is_file():
    with zipfile.ZipFile(zip_candidate) as z: z.extractall(WORK_DATA_DIR)
else:
    raise FileNotFoundError(f"Couldn't find dataset at {src}")

# Flatten single top-level wrapper folder if present.
top = list(WORK_DATA_DIR.iterdir())
if len(top) == 1 and top[0].is_dir():
    inner = top[0]
    for item in inner.iterdir():
        shutil.move(str(item), str(WORK_DATA_DIR / item.name))
    inner.rmdir()

print("unpacked. contents:")
for p in sorted(WORK_DATA_DIR.iterdir()): print(" ", p.name)

## 4. Dataset audit — class balance + names integrity

Counts instances and images per class across train+val, prints the dataset's `names:` order, and flags whether it matches the runtime.

In [ ]:
import yaml
from collections import Counter

# Try to find data.yaml; if missing, fall back to a classes.txt.
dyaml = WORK_DATA_DIR / "data.yaml"
dataset_names = None
if dyaml.exists():
    src_cfg = yaml.safe_load(dyaml.read_text())
    dataset_names = src_cfg.get("names")
    if isinstance(dataset_names, dict):
        # data.yaml can store names as dict {0: 'bird', 1: ...} — normalize to list
        dataset_names = [dataset_names[k] for k in sorted(dataset_names)]
elif (WORK_DATA_DIR / "classes.txt").exists():
    dataset_names = [l.strip() for l in (WORK_DATA_DIR / "classes.txt").read_text().splitlines() if l.strip()]

print("=== names in dataset ===")
print(dataset_names)
print("=== runtime names ===")
print(RUNTIME_CLASS_NAMES)

def names_equivalent(a, b):
    """Match case-insensitively, ignoring underscores/dashes (shahed_136 vs Shahed-136)."""
    norm = lambda s: str(s).lower().replace("_", "").replace("-", "").strip()
    return [norm(x) for x in (a or [])] == [norm(x) for x in (b or [])]

names_match = names_equivalent(dataset_names, RUNTIME_CLASS_NAMES)
print("\n>>> names order MATCHES runtime? ", names_match)

# Walk every .txt label file under WORK_DATA_DIR to count class instances
instance_counts = Counter()
image_counts    = Counter()
tiny_dji_count  = 0  # bboxes whose area is <0.001 of image area
labels_seen     = 0
for label_path in WORK_DATA_DIR.rglob("*.txt"):
    # Skip readmes, classes.txt, etc.
    if label_path.name == "classes.txt":
        continue
    try:
        text = label_path.read_text(errors="ignore")
    except Exception:
        continue
    classes_in_file = set()
    for line in text.splitlines():
        parts = line.split()
        if len(parts) < 5: continue
        try:
            cid = int(parts[0])
            w = float(parts[3]); h = float(parts[4])
        except ValueError:
            continue
        instance_counts[cid] += 1
        classes_in_file.add(cid)
        if cid == DJI_CLASS_ID and (w * h) < 0.001:
            tiny_dji_count += 1
    for c in classes_in_file:
        image_counts[c] += 1
    labels_seen += 1

print(f"\n=== scanned {labels_seen} label files ===")
print("\nclass | name (dataset)        | instances | images")
print("------+-----------------------+-----------+--------")
max_classes = max(len(dataset_names or []), len(RUNTIME_CLASS_NAMES), max(instance_counts.keys() | image_counts.keys() | {0}) + 1)
for cid in range(max_classes):
    name = (dataset_names[cid] if dataset_names and cid < len(dataset_names) else "<unknown>")
    print(f"  {cid:3d} | {name:21s} | {instance_counts[cid]:9d} | {image_counts[cid]:6d}")

if instance_counts.get(DJI_CLASS_ID, 0):
    print(f"\nDJI tiny-bbox count (area < 0.1% of image): {tiny_dji_count} / {instance_counts[DJI_CLASS_ID]} "
          f"({100 * tiny_dji_count / max(1, instance_counts[DJI_CLASS_ID]):.1f}%)")
    print("Tiny bboxes look like birds — if this ratio is high, expect Bird/DJI confusion regardless of training.")

# Stash for §5 / §6
_audit_dataset_names = dataset_names
_audit_names_match   = names_match
_audit_image_counts  = image_counts

## 5. Auto-remap class IDs if the dataset's `names:` order doesn't match the runtime

If the audit above prints `names order MATCHES runtime? False`, this cell finds the per-class index mapping and rewrites every `.txt` label so class IDs now line up with the runtime. Idempotent — running it twice when names already match is a no-op.

In [ ]:
if _audit_names_match:
    print("names already match runtime — skipping remap.")
else:
    if not _audit_dataset_names:
        raise RuntimeError("Cannot remap: dataset names are unknown. Add a classes.txt or fix data.yaml first.")

    norm = lambda s: str(s).lower().replace("_", "").replace("-", "").strip()
    runtime_norm = [norm(n) for n in RUNTIME_CLASS_NAMES]
    dataset_norm = [norm(n) for n in _audit_dataset_names]

    # old_id -> new_id mapping
    remap = {}
    missing = []
    for old_id, dn in enumerate(dataset_norm):
        if dn in runtime_norm:
            remap[old_id] = runtime_norm.index(dn)
        else:
            missing.append((old_id, _audit_dataset_names[old_id]))
    print("REMAP plan (old -> new):")
    for k, v in sorted(remap.items()):
        print(f"  {k:2d} ({_audit_dataset_names[k]:>15s})  ->  {v:2d} ({RUNTIME_CLASS_NAMES[v]})")
    if missing:
        print("WARNING — dataset has classes the runtime doesn't know about:")
        for old_id, name in missing:
            print(f"  old_id={old_id} name={name}")
        print("Those instances will be DROPPED.")

    # Rewrite every label file in place.
    rewritten = 0
    dropped_lines = 0
    for label_path in WORK_DATA_DIR.rglob("*.txt"):
        if label_path.name == "classes.txt":
            continue
        new_lines = []
        changed = False
        for line in label_path.read_text(errors="ignore").splitlines():
            parts = line.split()
            if len(parts) < 5:
                new_lines.append(line); continue
            try:
                old_id = int(parts[0])
            except ValueError:
                new_lines.append(line); continue
            if old_id not in remap:
                dropped_lines += 1
                changed = True
                continue
            new_id = remap[old_id]
            if new_id != old_id:
                changed = True
            new_lines.append(" ".join([str(new_id)] + parts[1:]))
        if changed:
            label_path.write_text("\n".join(new_lines) + ("\n" if new_lines else ""))
            rewritten += 1

    # Update / rewrite classes.txt and data.yaml so downstream Ultralytics
    # always sees the runtime order.
    (WORK_DATA_DIR / "classes.txt").write_text("\n".join(RUNTIME_CLASS_NAMES) + "\n")
    print(f"\nrewrote {rewritten} label files, dropped {dropped_lines} unmappable lines")
    print("updated classes.txt to runtime order.")

## 6. Oversample DJI if it's underrepresented

Duplicates DJI-containing image/label pairs (with `_djiosN` suffix) until DJI's image count reaches `DJI_OVERSAMPLE_TARGET_RATIO` × the largest class's image count. Train-split only — never inflates val.

In [ ]:
from collections import Counter
import shutil

def find_train_dirs():
    for d in [WORK_DATA_DIR / "train" / "images", WORK_DATA_DIR / "train"]:
        labels = d.parent / "labels" if d.name == "images" else d / "labels"
        if d.is_dir() and labels.is_dir():
            return d, labels
    return None, None

img_dir, lbl_dir = find_train_dirs()
if img_dir is None:
    print("No train/images + train/labels pair found — skipping oversample.")
else:
    # Recount per-class IMAGES on train split only.
    image_counts = Counter()
    dji_label_files = []
    for lp in lbl_dir.glob("*.txt"):
        classes_in_file = set()
        for line in lp.read_text(errors="ignore").splitlines():
            parts = line.split()
            if len(parts) < 5: continue
            try: classes_in_file.add(int(parts[0]))
            except ValueError: continue
        for c in classes_in_file:
            image_counts[c] += 1
        if DJI_CLASS_ID in classes_in_file:
            dji_label_files.append(lp)

    print("train per-class images:", dict(sorted(image_counts.items())))
    print(f"DJI images: {image_counts[DJI_CLASS_ID]}")

    max_other = max((c for cid, c in image_counts.items() if cid != DJI_CLASS_ID), default=0)
    target = int(max_other * DJI_OVERSAMPLE_TARGET_RATIO)
    needed = max(0, target - image_counts[DJI_CLASS_ID])
    print(f"target DJI images: {target}  (ratio={DJI_OVERSAMPLE_TARGET_RATIO} of largest other class {max_other})")
    print(f"need to add: {needed}")

    if needed > 0 and dji_label_files:
        added = 0
        round_idx = 1
        while added < needed:
            for lp in dji_label_files:
                if added >= needed: break
                # Find the matching image file (any common extension)
                stem = lp.stem
                img_match = None
                for ext in (".jpg", ".jpeg", ".png", ".bmp", ".webp"):
                    candidate = img_dir / f"{stem}{ext}"
                    if candidate.exists():
                        img_match = candidate; break
                if img_match is None:
                    continue
                new_stem = f"{stem}_djios{round_idx}"
                shutil.copy2(img_match, img_dir / f"{new_stem}{img_match.suffix}")
                shutil.copy2(lp,       lbl_dir / f"{new_stem}.txt")
                added += 1
            round_idx += 1
            if round_idx > 50: break  # safety
        print(f"oversampled DJI: added {added} duplicate image/label pairs")
    else:
        print("DJI is already balanced — no oversampling.")

## 7. Build / repair `data.yaml`

Always writes the RUNTIME class order — by this point all labels have been remapped to match.

In [ ]:
import yaml
DATA_YAML = WORK_DATA_DIR / "data.yaml"

def find_split(name):
    cands = [WORK_DATA_DIR / name / "images", WORK_DATA_DIR / name]
    if name == "val":
        cands = [WORK_DATA_DIR / "val" / "images", WORK_DATA_DIR / "valid" / "images",
                 WORK_DATA_DIR / "val", WORK_DATA_DIR / "valid"] + cands
    for c in cands:
        if c.is_dir(): return c
    return None

cfg = {
    "path":  str(WORK_DATA_DIR),
    "nc":    len(RUNTIME_CLASS_NAMES),
    "names": RUNTIME_CLASS_NAMES,
}
for split in ("train", "val", "test"):
    found = find_split(split)
    if found:
        cfg[split] = str(found); print(f"  {split:5s} -> {found}")

DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
print("\n--- data.yaml ---\n" + DATA_YAML.read_text())

## 8. Train — class-separation focused

Why these settings beat v4 for DJI:

| param | v4 (regressed)   | v5 (this run)    | reason |
|-------|------------------|------------------|--------|
| `mixup`      | 0.20  | **0.0**  | mixup linearly interpolates *labels* — a 50/50 Bird+DJI image trains both heads toward the average. Lethal for Bird/DJI separation. |
| `copy_paste` | 0.15  | **0.05** | still some augmentation but won't paste a DJI onto a frame already labeled as Bird (and vice versa). |
| `cls`        | 0.4   | **0.6**  | raise classification-loss weight so the model is rewarded for keeping classes apart, not for box quality alone. |
| `imgsz`      | 1280  | 1280     | keep — tiny DJI bboxes need pixels. |
| `box`        | 8.0   | 8.0      | keep — good for recall on small targets. |
| `mosaic`     | 1.0   | 1.0      | keep — multi-object scenes per batch. |

Also `close_mosaic=15` to disable mosaic for the final 15 epochs, letting the model settle on clean single-class images.

In [ ]:
from ultralytics import YOLO

last_pt = DRIVE_ROOT / "runs" / RUN_NAME / "weights" / "last.pt"
resume_active = RESUME and last_pt.exists()

if resume_active:
    print(f"RESUMING from: {last_pt}")
    start_weights = str(last_pt)
elif PREV_BEST_PT and Path(PREV_BEST_PT).exists():
    print(f"fresh fine-tune from: {PREV_BEST_PT}")
    start_weights = PREV_BEST_PT
else:
    print(f"fresh from base: {BASE_MODEL}")
    start_weights = BASE_MODEL

model = YOLO(start_weights)

train_kwargs = dict(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    name=RUN_NAME,
    project=str(DRIVE_ROOT / "runs"),
    save_period=SAVE_PERIOD,
    patience=PATIENCE,
    plots=True,
    # === class-separation knobs ===
    box=8.0,
    cls=0.6,            # was 0.4 in v4 — RAISE to keep classes apart
    dfl=1.5,
    mosaic=1.0,
    close_mosaic=15,
    mixup=0.0,          # was 0.20 in v4 — KILL label interpolation
    copy_paste=0.05,    # was 0.15 in v4 — gentle, won't cross-label
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5, translate=0.10, scale=0.55,
    fliplr=0.5,
    # === schedule ===
    lr0=0.005,
    cos_lr=True,
    warmup_epochs=5.0,
    cache=False,
    device=0,
    seed=42,
    exist_ok=True,
)
if resume_active:
    train_kwargs["resume"] = True

results = model.train(**train_kwargs)
print("training complete")

## 9. Evaluate — per-class breakdown (DJI called out)

The headline mAP can mask one bad class. We pull `metrics.box.maps` which is per-class mAP@0.5:0.95, plus per-class P/R, and highlight DJI explicitly.

In [ ]:
run_dir = Path(model.trainer.save_dir)
best_pt = run_dir / "weights" / "best.pt"

eval_model = YOLO(str(best_pt))
val_metrics = eval_model.val(data=str(DATA_YAML), split="val", imgsz=IMG_SIZE, plots=True, save_json=True)

import pandas as pd

# Per-class arrays from Ultralytics: maps, p, r are vectors aligned with class index.
rows = []
for cid, cname in enumerate(RUNTIME_CLASS_NAMES):
    rows.append({
        "class_id":     cid,
        "name":         cname,
        "precision":    round(float(val_metrics.box.p[cid]), 4) if cid < len(val_metrics.box.p) else None,
        "recall":       round(float(val_metrics.box.r[cid]), 4) if cid < len(val_metrics.box.r) else None,
        "mAP@0.5":      round(float(val_metrics.box.ap50[cid]), 4) if cid < len(val_metrics.box.ap50) else None,
        "mAP@0.5:0.95": round(float(val_metrics.box.maps[cid]), 4) if cid < len(val_metrics.box.maps) else None,
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(run_dir / "per_class_metrics.csv", index=False)

dji_row = df[df["class_id"] == DJI_CLASS_ID].iloc[0]
print("\n=== DJI verdict ===")
print(f"precision: {dji_row['precision']}   recall: {dji_row['recall']}   mAP50: {dji_row['mAP@0.5']}")
if dji_row["recall"] is not None and dji_row["recall"] < 0.6:
    print("DJI recall is below 0.6 — model still struggles. See §10/§11 confusion matrix for what it's confusing DJI with.")
elif dji_row["recall"] is not None and dji_row["recall"] >= 0.85:
    print("DJI is in good shape — ship best.pt.")

## 10. Plots — confusion matrix, PR curves, training curves

In [ ]:
from IPython.display import Image, display
for fname in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png",
              "PR_curve.png", "P_curve.png", "R_curve.png", "F1_curve.png",
              "labels.jpg", "val_batch0_pred.jpg"]:
    p = run_dir / fname
    if p.exists():
        print(fname); display(Image(str(p)))

## 11. DJI-focused sample predictions

Find val images that actually contain DJI and run the new model on them — the ground-truth test for whether DJI is now detected as DJI (not Bird).

In [ ]:
import cv2, matplotlib.pyplot as plt, random, yaml as _yaml
cfg = _yaml.safe_load(DATA_YAML.read_text())
val_img_dir = Path(cfg["val"])
val_lbl_dir = val_img_dir.parent / "labels" if val_img_dir.name == "images" else val_img_dir / "labels"

dji_val_imgs = []
for lp in val_lbl_dir.glob("*.txt"):
    for line in lp.read_text(errors="ignore").splitlines():
        parts = line.split()
        if len(parts) >= 5 and parts[0] == str(DJI_CLASS_ID):
            stem = lp.stem
            for ext in (".jpg", ".jpeg", ".png", ".bmp", ".webp"):
                candidate = val_img_dir / f"{stem}{ext}"
                if candidate.exists():
                    dji_val_imgs.append(candidate); break
            break

print(f"val images containing DJI: {len(dji_val_imgs)}")
if dji_val_imgs:
    sample = random.sample(dji_val_imgs, min(6, len(dji_val_imgs)))
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, img_path in zip(axes.flat, sample):
        res = eval_model.predict(source=str(img_path), imgsz=IMG_SIZE, conf=0.20, augment=True, verbose=False)[0]
        plotted = res.plot()
        ax.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
        ax.set_title(img_path.name, fontsize=9); ax.axis('off')
    plt.tight_layout(); plt.show()

## 12. Save model — top-level + timestamped

In [ ]:
import shutil, datetime as dt

stamp = dt.datetime.now().strftime("%Y%m%d_%H%M")
top_best = DRIVE_ROOT / "best.pt"
stamped  = DRIVE_ROOT / f"best_{RUN_NAME}_{stamp}.pt"

shutil.copy2(best_pt, top_best)
shutil.copy2(best_pt, stamped)

print("saved:")
print("  ", top_best,  "  (overwrites the live one used by the backend)")
print("  ", stamped,  "  (timestamped backup)")

## What to look at after training

1. **§9 per-class table** — the DJI row should show precision ≥ 0.85 and recall ≥ 0.85. If recall is below 0.6, the model is still calling DJI a Bird.
2. **§10 `confusion_matrix.png`** — look at row `dji`. If a big chunk lands in column `Bird`, the model still confuses them. Next-step knobs in that case:
   - Drop `mosaic` to 0.5 (mosaic at full strength can put DJI on the same canvas as a Bird every epoch)
   - Raise `cls` further to 0.8
   - Increase `DJI_OVERSAMPLE_TARGET_RATIO` to 1.0
3. **§11 sample predictions** — these are the visual ground truth. If DJI consistently gets a `dji` label box, you're done.

Once happy, copy `final_model/best.pt` from Drive to `capstone/models/best.pt` locally, restart uvicorn — the backend's startup log will print the class map again so you can confirm `5: dji` is still there.